# HW03 — Честный ML-эксперимент / A valid ML experiment

100 баллов: код 80 + объяснение 20. / 100 points: code 80 + explanation 20.

Откройте в Colab на CPU. Заполните ФИО/группу, замените TODO, перезапустите сеанс и выполните все ячейки. Сдайте `.ipynb` с выводами. / Open in Colab on CPU, enter your name/group, replace TODOs, restart and run all cells, and submit the executed `.ipynb`.

Срок и штраф определяет преподаватель по подтверждённому времени сдачи; локальные часы не используются. / The instructor calculates deadline penalties using verified submission time, never the notebook clock.

Готовые функции sklearn разрешены в T3 и эксперименте. T1–T2 реализуйте через NumPy. Список кандидатов и правило равенств не меняйте. / sklearn is allowed in T3 and the experiment. Use NumPy for T1–T2. Keep the candidate list and tie rule unchanged.

In [ ]:
STUDENT_NAME = ""
STUDENT_GROUP = ""

In [ ]:
import json, hashlib, platform
import numpy as np
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

In [ ]:
# Учебные синтетические данные / Synthetic teaching data
rng = np.random.default_rng(2026)
groups = np.repeat(np.arange(60), 4)
latent = rng.normal(size=(60, 3))
X = latent[groups] + rng.normal(0, .12, size=(240, 3))
X *= np.array([1., 100., .01])
y = (latent[groups, 0] + .6 * latent[groups, 1] > 0).astype(int)
row_ids = np.arange(len(y))
assert X.shape == (240, 3) and set(np.unique(y)) == {0, 1}

## T1 — 20 баллов / points
`split_groups(groups, seed=42)` возвращает `(train_idx, val_idx, test_idx)` — возрастающие индексы строк. Непустой 1D массив целочисленных групп, не менее 5 уникальных групп. Возьмите `np.unique`, переставьте через `np.random.default_rng(seed).permutation`. Первые `ceil(0.2*G)` групп — Test, следующие столько же — Validation, остаток — Train. Каждая строка ровно в одной части. / Return sorted row-index arrays. Input: a nonempty 1D array of integer group IDs with at least 5 distinct groups. Permute `np.unique(groups)` using the specified generator; first ceil(0.2*G) groups go to Test, next the same count to Validation, remaining to Train. Every row appears exactly once.

In [ ]:
def split_groups(groups, seed=42):
    raise NotImplementedError("TODO")

## T2 — 20 баллов / points
`fit_standardizer(X_train)` возвращает среднее и стандартное отклонение по колонкам (`ddof=0`). Нулевое отклонение заменяется на 1. `apply_standardizer(X, mean, scale)` применяет готовые статистики, ничего не переобучая. Входы конечные, 2D, непустые, одинаковое число признаков. / Return per-column means and population standard deviations; replace zero scales with 1. Apply those statistics without refitting. Inputs are finite nonempty 2D arrays with matching feature counts.

In [ ]:
def fit_standardizer(X_train):
    raise NotImplementedError("TODO")

def apply_standardizer(X, mean, scale):
    raise NotImplementedError("TODO")

## T3 — 15 баллов / points
`make_model(C)` возвращает необученный `Pipeline`: `scale=StandardScaler()`, `model=LogisticRegression(C=C, solver="lbfgs", max_iter=1000, random_state=42)`. C положителен. Не вызывайте fit внутри функции. / Return the unfitted pipeline with the exact named steps and settings above. C is positive. Do not call fit inside the function.

In [ ]:
def make_model(C):
    raise NotImplementedError("TODO")

## T4 — 15 баллов / points
`choose_candidate(records)` возвращает C записи с наибольшим `val_f1`. При точном равенстве — меньший C. Список непустой; записи содержат только C>0 и конечный val_f1 в [0,1]. Не изменяйте вход. / Return C with the highest validation macro-F1; exact ties choose the smaller C. The list is nonempty with valid C and finite scores. Do not mutate it.

In [ ]:
def choose_candidate(records):
    raise NotImplementedError("TODO")

## T5 — 10 баллов / points
`make_log(train_ids, val_ids, test_ids, seed, selected_C, val_f1, test_f1)` возвращает JSON-сериализуемый dict: assignment="HW03_Experiment", seed, train_ids, val_ids, test_ids, selected_C, val_f1, test_f1, numpy_version, sklearn_version. ID сохраняйте в исходном порядке как Python int. / Return a JSON-serializable dictionary with these exact keys; preserve ID order as Python ints. Include actual installed library versions.

In [ ]:
def make_log(train_ids, val_ids, test_ids, seed, selected_C, val_f1, test_f1):
    raise NotImplementedError("TODO")

## Эксперимент / Experiment
После реализации функций выполните сравнение. Не меняйте разбиение, список C или метрику по результату Test. Если результат хуже baseline, корректно объясните его — высокий балл метрики не является условием получения баллов за ДЗ. / Run after implementing the functions. Do not change splits, C candidates, or metric after inspecting Test. Explain a result below baseline honestly; a high metric value is not required for homework credit.

In [ ]:
try:
    train, val, test = split_groups(groups, seed=42)
    for a, b in [(train, val), (train, test), (val, test)]:
        assert not set(groups[a]) & set(groups[b])
    for part in (train, val, test):
        assert set(np.unique(y[part])) == {0, 1}, 'Need both classes for this teaching example'
    baseline = DummyClassifier(strategy='most_frequent').fit(X[train], y[train])
    print('Baseline validation macro-F1:', f1_score(y[val], baseline.predict(X[val]), average='macro', zero_division=0))
    records, models = [], {}
    for C in [.1, 1., 10.]:
        model = make_model(C).fit(X[train], y[train])
        models[C] = model
        records.append({'C':C, 'val_f1':float(f1_score(y[val], model.predict(X[val]), average='macro', zero_division=0))})
    selected_C = choose_candidate(records)
    print('Validation results:', records, 'Selected C:', selected_C)
    # Выбор зафиксирован ДО доступа к Test / Freeze selection BEFORE using Test
    selected = models[selected_C]
    test_f1 = f1_score(y[test], selected.predict(X[test]), average='macro', zero_division=0)
    baseline_test = f1_score(y[test], baseline.predict(X[test]), average='macro', zero_division=0)
    log = make_log(row_ids[train], row_ids[val], row_ids[test], 42, selected_C,
                   next(r['val_f1'] for r in records if r['C'] == selected_C), test_f1)
    log['baseline_test_f1'] = float(baseline_test)
    log['data_sha256'] = hashlib.sha256(X.tobytes()+y.tobytes()+groups.tobytes()).hexdigest()
    print(json.dumps(log, ensure_ascii=False))

except NotImplementedError:
    print("Сначала завершите T1–T5 / Complete T1–T5 first")

## Объяснение — 20 баллов / Explanation — 20 points
1. Приведите число строк/групп/долю класса 1 в каждой части и объясните выбор групповой единицы разбиения (5). / Give row counts, group counts, and positive-class proportions per split; justify group splitting (5).
2. Для Train=[0,2], Validation=[100] вручную вычислите mean, scale, transform(100) и объясните утечку при fit на всём наборе (5). / Compute the training mean, scale, transformed validation value; explain full-data fitting leakage (5).
3. Запишите таблицу кандидатов, выбранный C, результаты модели и baseline на Test; объясните решение и одно ограничение синтетических данных (6). / Report all validation candidates, selected C, model and baseline test scores; explain selection and one synthetic-data limitation (6).
4. Укажите источник времени сдачи (без выдуманной даты), источники/AI-помощь, изменяемую гипотезу следующего эксперимента (4). / State the submission-time source (do not invent a timestamp), sources/AI assistance, and one hypothesis for the next experiment (4).

**Ваш ответ / Your response:**

…

## Самопроверка / Self-check
Не изменяйте тесты. `score` — только код из 80, не итог из 100. Преподаватель отдельно проверяет объяснение, метод и корректность эксперимента. / Do not edit tests. `score` is code credit out of 80, not the final grade out of 100. Explanations, methods, and experiment validity require instructor review.

In [ ]:
def check_t1():
    for g in [np.repeat(np.arange(10), [1,2,3,4,5,1,2,3,4,5]), np.array([99,3,7,3,8,9,99,8,9,7])]:
        original = g.copy()
        for seed in [42, 7]:
            parts = split_groups(g, seed)
            assert len(parts) == 3
            u = np.random.default_rng(seed).permutation(np.unique(g))
            n = int(np.ceil(.2*len(u)))
            expected = [u[2*n:], u[n:2*n], u[:n]]
            for part, eg in zip(parts, expected):
                assert np.array_equal(part, np.flatnonzero(np.isin(g, eg)))
            assert np.array_equal(np.sort(np.concatenate(parts)), np.arange(len(g)))
        assert np.array_equal(g, original)
def check_t2():
    a = np.array([[0.,7.,-2.],[2.,7.,2.]])
    mean, scale = fit_standardizer(a)
    assert np.allclose(mean,[1,7,0]) and np.allclose(scale,[1,1,2])
    out = apply_standardizer([[100.,7.,4.]], mean, scale)
    assert np.allclose(out, [[99.,0.,2.]]) and np.allclose(a, [[0,7,-2],[2,7,2]])
    mean, scale = fit_standardizer([[5.,9.]])
    assert np.allclose(scale,[1,1])
def check_t3():
    m = make_model(.25)
    assert isinstance(m, Pipeline) and list(m.named_steps) == ['scale','model']
    assert isinstance(m['scale'], StandardScaler) and isinstance(m['model'], LogisticRegression)
    assert m['model'].C == .25 and m['model'].max_iter == 1000 and m['model'].solver == 'lbfgs'
    assert m['model'].random_state == 42 and not hasattr(m['scale'],'mean_')
    m.fit([[0,10],[1,20],[3,40],[4,50]], [0,0,1,1])
    assert np.allclose(m['scale'].mean_,[2,30])
    assert m.predict([[2,30]]).shape == (1,)
def check_t4():
    r=[{'C':10.,'val_f1':.8},{'C':1.,'val_f1':.8},{'C':.1,'val_f1':.7}]
    old=json.dumps(r)
    assert choose_candidate(r)==1. and json.dumps(r)==old
    assert choose_candidate([{'C':.1,'val_f1':.1},{'C':10.,'val_f1':.9}])==10.
    assert choose_candidate([{'C':2.,'val_f1':0.}])==2.
def check_t5():
    d=make_log(np.array([3,1]),np.array([2]),np.array([0]),np.int64(42),np.float64(.1),.6,.5)
    json.dumps(d,allow_nan=False)
    assert d['assignment']=='HW03_Experiment' and d['train_ids']==[3,1]
    assert d['val_ids']==[2] and d['test_ids']==[0] and d['seed']==42
    assert d['selected_C']==.1 and d['val_f1']==.6 and d['test_f1']==.5
    assert d['numpy_version']==np.__version__ and d['sklearn_version']==sklearn.__version__
checks=[('T1',20,check_t1),('T2',20,check_t2),('T3',15,check_t3),('T4',15,check_t4),('T5',10,check_t5)]
results=[]
for task,points,fn in checks:
    try:
        fn(); results.append({'task':task,'points':points,'max_points':points,'status':'OK'})
    except Exception as e:
        results.append({'task':task,'points':0,'max_points':points,'status':type(e).__name__})
print(json.dumps({'name':STUDENT_NAME,'group':STUDENT_GROUP,'assignment':'HW03_Experiment',
 'score':sum(r['points'] for r in results),'score_kind':'code_only','code_score':sum(r['points'] for r in results),
 'code_max':80,'explanation_score':None,'total_score':None,'max_score':100,'tasks':results},ensure_ascii=False))